In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, GRU
from tensorflow.keras.optimizers import Adam
import random
import sys # For sys.stdout.flush()
import requests # For downloading from URL
import warnings # For warning suppression
import os # To check if a file exists locally

print("--- RNN for Text Generation ---")
print(f"TensorFlow Version: {tf.__version__}")

# --- Configuration ---
# Set this to the path of your local Shakespeare file.
# If you are running in Google Colab, '/content/shakespeare.txt' is a common path
# if you've uploaded it to your Colab session.
# If you are running locally, it might just be 'shakespeare.txt' if it's in the same directory.
# If you want to download from the web, change this back to the URL.
TEXT_SOURCE = '/content/shakespeare.txt' # This will be treated as a local path first

# Define a local filename where the text will be saved/read from.
# This is useful even if TEXT_SOURCE is a URL, as it saves the downloaded content.
LOCAL_TEXT_FILE = 'shakespeare_processed.txt' # Changed name slightly to avoid confusion

# --- IMPORTANT CHANGE FOR PERFORMANCE ---
TEXT_SAMPLE_SIZE = 500000 # Use first N characters. Reduce this further for faster runs (e.g., 50000 or 100000).
# Set to None to use the entire file.

SEQUENCE_LENGTH = 100
STEP_SIZE = 3
EMBEDDING_DIM = 128
LSTM_UNITS = 256
DROPOUT_RATE = 0.2
EPOCHS = 30 # You might want to reduce this to 5-10 for faster testing, especially on CPU
BATCH_SIZE = 128
GENERATION_LENGTH = 500
TEMPERATURE = 0.7

# --- 1. Load and Prepare Text Data ---
print("\n--- 1. Loading and Preparing Text Data ---")

full_text = ""
if TEXT_SOURCE.startswith('http://') or TEXT_SOURCE.startswith('https://'):
    print(f"Attempting to download text from URL: {TEXT_SOURCE}")
    try:
        response = requests.get(TEXT_SOURCE)
        response.raise_for_status() # Raise an exception for HTTP errors
        full_text = response.text.lower()
        print(f"Successfully downloaded text from URL. Full length: {len(full_text)} characters.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading text from URL: {e}")
        print(f"Please check your internet connection or the URL: {TEXT_SOURCE}")
        sys.exit(1)
else: # Assume it's a local file path
    print(f"Attempting to read text from local file: {TEXT_SOURCE}")
    try:
        with open(TEXT_SOURCE, 'r', encoding='utf-8') as f:
            full_text = f.read().lower()
        print(f"Successfully read text from local file. Full length: {len(full_text)} characters.")
    except FileNotFoundError:
        print(f"Error: Local file '{TEXT_SOURCE}' not found.")
        print("Please ensure the file exists at the specified path or change TEXT_SOURCE to a valid URL.")
        sys.exit(1)
    except Exception as e:
        print(f"An error occurred while reading the local file: {e}")
        sys.exit(1)

# Apply the sample size limit after loading (whether from URL or local file)
if TEXT_SAMPLE_SIZE is not None and len(full_text) > TEXT_SAMPLE_SIZE:
    text = full_text[:TEXT_SAMPLE_SIZE]
    print(f"Using first {len(text)} characters for training (sampled from original).")
else:
    text = full_text
    print(f"Using full text of {len(text)} characters.")

# Optional: Save the *processed/sampled* text to a standardized local file
try:
    with open(LOCAL_TEXT_FILE, 'w', encoding='utf-8') as f:
        f.write(text)
    print(f"Processed text saved to '{LOCAL_TEXT_FILE}'.")
except Exception as e:
    print(f"Warning: Could not save processed text to '{LOCAL_TEXT_FILE}': {e}")


# Create character vocabulary
chars = sorted(list(set(text)))
char_to_int = {char: i for i, char in enumerate(chars)}
int_to_char = {i: char for i, char in enumerate(chars)}
VOCAB_SIZE = len(chars)

print(f"Total unique characters (vocabulary size): {VOCAB_SIZE}")
print(f"First 10 characters in vocabulary: {chars[:10]}")

print(f"Creating sequences of length {SEQUENCE_LENGTH} with step size {STEP_SIZE}...")
input_sequences = []
output_characters = []
for i in range(0, len(text) - SEQUENCE_LENGTH, STEP_SIZE):
    input_sequences.append(text[i:i + SEQUENCE_LENGTH])
    output_characters.append(text[i + SEQUENCE_LENGTH])

print(f"Total sequences created: {len(input_sequences)}")

X = np.zeros((len(input_sequences), SEQUENCE_LENGTH), dtype=np.int32)
y = np.zeros((len(input_sequences), VOCAB_SIZE), dtype=np.int32)

for i, sequence in enumerate(input_sequences):
    for t, char in enumerate(sequence):
        X[i, t] = char_to_int[char]
    y[i, char_to_int[output_characters[i]]] = 1

print(f"Shape of input data (X): {X.shape}")
print(f"Shape of output data (y): {y.shape}")
print("Text data preparation complete.")

# --- Helper function for sampling a character ---
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

# --- 2. Build the RNN Model (LSTM) ---
print("\n--- 2. Building RNN Model (LSTM) ---")

# Globally ignore the `input_length` warning from Keras Embedding layer
warnings.filterwarnings('ignore', message='Argument `input_length` is deprecated. Just remove it.', category=UserWarning)

def build_lstm_model():
    model = Sequential([
        Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM), # input_length removed
        LSTM(LSTM_UNITS, return_sequences=True, dropout=DROPOUT_RATE, recurrent_dropout=DROPOUT_RATE),
        LSTM(LSTM_UNITS, dropout=DROPOUT_RATE, recurrent_dropout=DROPOUT_RATE),
        Dense(VOCAB_SIZE, activation='softmax')
    ])
    return model

model_lstm = build_lstm_model()
model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()
print("LSTM model built and compiled.")

# --- 3. Train the Model (LSTM) ---
print(f"\n--- 3. Training LSTM Model for {EPOCHS} epochs ---")
history_lstm = model_lstm.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
print("LSTM model training complete.")

# --- 4. Generate Text Samples (LSTM) ---
print(f"\n--- 4. Generating Text Samples with LSTM Model (Length: {GENERATION_LENGTH}) ---")
def generate_text(model, generation_length, temperature, seed_text=""):
    if not seed_text:
        start_index = random.randint(0, len(text) - SEQUENCE_LENGTH - 1)
        seed_text = text[start_index:start_index + SEQUENCE_LENGTH]

    generated_text = seed_text
    sys.stdout.write(f"Seed: '{seed_text}'\n")
    sys.stdout.write("Generated: ")
    sys.stdout.write(seed_text)

    for i in range(GENERATION_LENGTH):
        x_pred = np.zeros((1, SEQUENCE_LENGTH), dtype=np.int32)
        for t, char in enumerate(seed_text):
            if char not in char_to_int: # Handle potential OOV chars if seed is custom
                char = ' ' # Replace with a common char or handle as needed
            x_pred[0, t] = char_to_int[char]

        # Suppress TensorFlow retracing warnings here for prediction
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, module='tensorflow')
            warnings.filterwarnings("ignore", category=UserWarning, message=".*tf.function retracing.*")
            preds = model.predict(x_pred, verbose=0)[0]

        next_index = sample(preds, temperature)
        next_char = int_to_char[next_index]

        generated_text += next_char
        seed_text = seed_text[1:] + next_char

        sys.stdout.write(next_char)
        sys.stdout.flush()
    print("\n")

print("\n--- LSTM Generated Sample 1 (Default Temperature) ---")
generate_text(model_lstm, GENERATION_LENGTH, TEMPERATURE)

print("\n--- LSTM Generated Sample 2 (Higher Temperature - more creative) ---")
generate_text(model_lstm, GENERATION_LENGTH, TEMPERATURE * 1.2, seed_text="romeo, wherefore art thou ")

print("\n--- LSTM Generated Sample 3 (Lower Temperature - more conservative) ---")
generate_text(model_lstm, GENERATION_LENGTH, TEMPERATURE * 0.8)

# --- Advanced Requirement: Experiment with GRU layers ---
print("\n--- ADVANCED: Building and Training GRU Model ---")

def build_gru_model():
    model = Sequential([
        Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM), # input_length removed
        GRU(LSTM_UNITS, return_sequences=True, dropout=DROPOUT_RATE, recurrent_dropout=DROPOUT_RATE),
        GRU(LSTM_UNITS, dropout=DROPOUT_RATE, recurrent_dropout=DROPOUT_RATE),
        Dense(VOCAB_SIZE, activation='softmax')
    ])
    return model

model_gru = build_gru_model()
model_gru.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_gru.summary()
print("GRU model built and compiled.")

# Train the GRU model
print(f"\n--- Training GRU Model for {EPOCHS} epochs ---")
history_gru = model_gru.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
print("GRU model training complete.")

# Generate Text Samples (GRU)
print(f"\n--- Generating Text Samples with GRU Model (Length: {GENERATION_LENGTH}) ---")

print("\n--- GRU Generated Sample 1 (Default Temperature) ---")
generate_text(model_gru, GENERATION_LENGTH, TEMPERATURE)

print("\n--- GRU Generated Sample 2 (Higher Temperature) ---")
generate_text(model_gru, GENERATION_LENGTH, TEMPERATURE * 1.2, seed_text="what light through yonder window breaks?")

print("\n--- GRU Generated Sample 3 (Lower Temperature) ---")
generate_text(model_gru, GENERATION_LENGTH, TEMPERATURE * 0.8)

# --- Comparison of Performance (Qualitative and Quantitative) ---
print("\n--- Performance Comparison ---")
print("Qualitative Observations:")
print("- Compare the coherence, grammar, and style of the generated text samples from both models.")
print("- LSTMs and GRUs are both effective for sequence tasks. GRUs often train faster due to their simpler architecture.")
print("- The 'temperature' parameter significantly affects creativity vs. coherence. Lower temperature results in more predictable, sometimes repetitive text, while higher temperature yields more surprising but potentially nonsensical output.")

print("\nQuantitative Observations (from training history):")
print(f"LSTM Final Training Loss: {history_lstm.history['loss'][-1]:.4f}")
print(f"LSTM Final Training Accuracy: {history_lstm.history['accuracy'][-1]:.4f}")
print(f"GRU Final Training Loss: {history_gru.history['loss'][-1]:.4f}")
print(f"GRU Final Training Accuracy: {history_gru.history['accuracy'][-1]:.4f}")
print("\nNote: Training time differences are harder to capture accurately without specific timing code,")
print("but GRUs typically have fewer parameters and can converge faster.")

print("\n--- Task Complete ---")

--- RNN for Text Generation ---
TensorFlow Version: 2.18.0

--- 1. Loading and Preparing Text Data ---
Attempting to read text from local file: /content/shakespeare.txt
Successfully read text from local file. Full length: 73553 characters.
Using full text of 73553 characters.
Processed text saved to 'shakespeare_processed.txt'.
Total unique characters (vocabulary size): 48
First 10 characters in vocabulary: ['\n', ' ', '!', "'", '(', ')', ',', '-', '.', '0']
Creating sequences of length 100 with step size 3...
Total sequences created: 24485
Shape of input data (X): (24485, 100)
Shape of output data (y): (24485, 48)
Text data preparation complete.

--- 2. Building RNN Model (LSTM) ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

GRU model built and compiled.

--- Training GRU Model for 30 epochs ---
Epoch 1/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 531s 3s/step - accuracy: 0.2515 - loss: 2.9286
Epoch 2/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 519s 3s/step - accuracy: 0.3906 - loss: 2.0908
Epoch 3/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 524s 3s/step - accuracy: 0.4330 - loss: 1.8999
Epoch 4/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 552s 3s/step - accuracy: 0.4746 - loss: 1.7728
Epoch 5/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 511s 3s/step - accuracy: 0.4910 - loss: 1.6720
Epoch 6/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 556s 3s/step - accuracy: 0.5209 - loss: 1.5854
Epoch 7/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 510s 3s/step - accuracy: 0.5361 - loss: 1.5099
Epoch 8/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 510s 3s/step - accuracy: 0.5445 - loss: 1.4562
Epoch 9/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 508s 3s/step - accuracy: 0.5659 - loss: 1.3756
Epoch 10/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 574s 3s/step - accuracy: 0.5747 - loss: 1.3448
Epoch 11/30
192/192 ━━━━━━━━━━━━━━━━━━━━ 556s 3s/step - a